# 01 · Warm start: baseline and retrieval variants (Chapter 7)

Reproduces **Table 7.8** (and the full-graph row of Table 7.7).

| Step | Thesis source |
|---|---|
| graph, features, split | `THIS_IS_THE_ONE_I_RUN` cells 6–12 |
| five training runs | same notebook, cells 24–33 (`run_variant`) |
| 5-draw strict evaluation | `rerun_hidden384` cells 2 and 5 |

The training logs print sklearn's **default** macro-F1 (what early stopping used). Report only the strict numbers from the evaluation section.
Known issues kept as in the thesis: hidden width = number of classes (212), label-based rare routing.

In [9]:
ON_COLAB  = False
REPO_DIR  = "/home/ppcrniai/Documents/ddi-linegraph"
DATA_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_data"
WORK_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_runs"
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [10]:
import os, sys
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
        raise SystemExit(f"Repo not found at {REPO_DIR}. Upload or clone it there first.")
    !bash {REPO_DIR}/setup/colab_setup.sh
# the editable install is only picked up by a fresh interpreter; make it importable now
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [11]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

  data_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_data
  work_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs
  raw_data_dir           /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data   (does not exist yet)
  drug_db                /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/Drug_META_DDIE.db   (does not exist yet)
  event_db               /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/event_META_DDIE_5foldLabel.db   (does not exist yet)
  graph_zip              /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip   (does not exist yet)
  graph_bin              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/ddi_graph_and_linegraph.bin   (does not exist yet)
  thesis_warm_ckpt_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_smart_fusion_whole_graph_checkpoints   (does not exist yet)
  thesis_hidden384_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_hidden384_experiment   (does not ex

In [12]:
from ddi import graph as G
from ddi.splits import filter_and_split_graph, save_split, check_split

if TEST_MODE:
    pair = os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin")
    assert os.path.exists(pair), "Run notebook 00 first to build the test subgraph."
    subgraph, line_subgraph = G.load_graph_pair(pair)
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    subgraph, line_subgraph = G.load_graph_pair(paths.graph_bin)   # 640M edges: high-RAM only
print(subgraph); print(line_subgraph)

line_subgraph = G.attach_endpoint_features(subgraph, line_subgraph)
g_lg_filt = filter_and_split_graph(line_subgraph, label_key="label")
del line_subgraph; gc.collect()

# write the split once, then check every later run against it
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.warm_dir, f"split_{RUN_TAG}.pt")
if os.path.exists(split_path):
    check_split(g_lg_filt, split_path)
else:
    save_split(g_lg_filt, split_path)
NUM_CLASSES = int(g_lg_filt.ndata["label"].max().item() + 1)
print("NUM_CLASSES:", NUM_CLASSES)

Graph(num_nodes=800, num_edges=81242,
      ndata_schemes={'orig_drug_id': Scheme(shape=(), dtype=torch.int64), 'id': Scheme(shape=(), dtype=torch.int64), 'smiles_embedding': Scheme(shape=(384,), dtype=torch.float32)}
      edata_schemes={'orig_edge_id': Scheme(shape=(), dtype=torch.int64), 'label': Scheme(shape=(), dtype=torch.int64), 'id': Scheme(shape=(), dtype=torch.int64), 'fused_embedding': Scheme(shape=(384,), dtype=torch.float32)})
Graph(num_nodes=81242, num_edges=8520237,
      ndata_schemes={}
      edata_schemes={})
Filtered graph: 81223 nodes | Classes kept: 123 (123 actually present)
Dropping 7 class(es) too rare in val/test split: [97, 120, 132, 137, 145, 164, 168]
Split summary: Train=56856 | Val=12180 | Test=12180
saved split -> /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/split_test800.pt
NUM_CLASSES: 181


## Train
Set `TRAIN = False` to skip training and only evaluate existing checkpoints.

In [13]:
from ddi import train

TRAIN = True
CKPT_DIR = os.path.join(paths.warm_dir, f"checkpoints_{RUN_TAG}")
KNN_DIR  = os.path.join(paths.warm_dir, f"knn_{RUN_TAG}")
os.makedirs(KNN_DIR, exist_ok=True)

run_args = dict(train.COMMON_RUN_ARGS)
if TEST_MODE:
    run_args["num_epochs"] = 2

results = {}
if TRAIN:
    for name, kw in train.WARM_START_VARIANTS.items():
        k = kw.get("proto_k")
        cache = os.path.join(KNN_DIR, f"knn_cache_k{k}.pt") if k else None
        _, m = train.run_variant(g_lg_filt, name, checkpoint_dir=CKPT_DIR, knn_cache_path=cache,
                                 device=device, **run_args, **kw)
        results[name] = m
        gc.collect(); torch.cuda.empty_cache()
    json.dump(results, open(os.path.join(paths.warm_dir, f"train_log_metrics_{RUN_TAG}.json"), "w"),
              indent=2, default=str)

Running variant: baseline
Training on 56856 nodes | Val=12180 | Test=12180
Sampler: uniform | Loss: ldam | Tail<thr=20: 105 classes
Checkpointing: save best only -> /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_baseline.pt | early_stopping_patience=6 | min_delta=0.0005


train(EVAL-epoch1) acc=0.3628 macroF1=0.0330 conf=0.703 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.3593 macroF1=0.0333 conf=0.704 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.0333@1 | no_improve=0/6 saved


train(EVAL-epoch2) acc=0.5546 macroF1=0.1034 conf=0.700 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.5409 macroF1=0.1057 conf=0.697 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.1057@2 | no_improve=0/6 saved
Loaded best model from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_baseline.pt (Val macroF1=0.1057, epoch=2)
=== Final Test (sklearn-default macro-F1; NOT the reporting convention) ===
Acc=0.5355 | MacroF1=0.1058 | MicroF1=0.5355 | Conf=0.696
TailF1(strict)=0.0000 | TailF1(present)=0.0000 | HeadF1(strict)=0.1621
Final model saved to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/final_model_baseline.pt

Done: baseline
{'acc': 0.5354679802955665, 'macro_f1': 0.10581838796957264, 'micro_f1': 0.5354679802955665, 'weighted_f1': 0.4962944530640157, 'avg_conf': 0.6963712573051453, 'best_val_macro_f1': 0.10567827373256734, 'best_epoch': 2, 'checkpoint_path': '/home/ppcrniai/Documents/

train(EVAL-epoch1) acc=0.4223 macroF1=0.0406 conf=0.631 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.4168 macroF1=0.0433 conf=0.630 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.0433@1 | no_improve=0/10 saved


train(EVAL-epoch2) acc=0.5357 macroF1=0.0976 conf=0.736 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.5184 macroF1=0.0986 conf=0.733 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.0986@2 | no_improve=0/10 saved
Loaded best model from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_proto_alpha_rare_only_k5.pt (Val macroF1=0.0986, epoch=2)
=== Final Test (sklearn-default macro-F1; NOT the reporting convention) ===
Acc=0.5195 | MacroF1=0.0982 | MicroF1=0.5195 | Conf=0.735
TailF1(strict)=0.0000 | TailF1(present)=0.0000 | HeadF1(strict)=0.1435
Final model saved to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/final_model_proto_alpha_rare_only_k5.pt

Done: proto_alpha_rare_only_k5
{'acc': 0.5195402298850574, 'macro_f1': 0.09818998409387607, 'micro_f1': 0.5195402298850574, 'weighted_f1': 0.47357762518981045, 'avg_conf': 0.7350683808326721, 'best_val_macro_f1': 0.09858221466022977, 'best_epoch'

train(EVAL-epoch1) acc=0.3871 macroF1=0.0404 conf=0.631 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.3807 macroF1=0.0414 conf=0.633 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.0414@1 | no_improve=0/10 saved


train(EVAL-epoch2) acc=0.5364 macroF1=0.1226 conf=0.658 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.5196 macroF1=0.1213 conf=0.653 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.1213@2 | no_improve=0/10 saved
Loaded best model from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_proto_alpha_rare_only_k3.pt (Val macroF1=0.1213, epoch=2)
=== Final Test (sklearn-default macro-F1; NOT the reporting convention) ===
Acc=0.5122 | MacroF1=0.1244 | MicroF1=0.5122 | Conf=0.653
TailF1(strict)=0.0000 | TailF1(present)=0.0000 | HeadF1(strict)=0.1843
Final model saved to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/final_model_proto_alpha_rare_only_k3.pt

Done: proto_alpha_rare_only_k3
{'acc': 0.5122331691297208, 'macro_f1': 0.12440382039583679, 'micro_f1': 0.5122331691297208, 'weighted_f1': 0.48725870588835696, 'avg_conf': 0.6533687114715576, 'best_val_macro_f1': 0.12127072855231943, 'best_epoch'

train(EVAL-epoch1) acc=0.4174 macroF1=0.0472 conf=0.647 | train_tail(strict)=0.0000 train_tail(present)=0.0000 | val acc=0.4108 macroF1=0.0493 conf=0.650 | val_tail(strict)=0.0000 val_tail(present)=0.0000 | best=0.0493@1 | no_improve=0/10 saved


train(EVAL-epoch2) acc=0.5512 macroF1=0.1728 conf=0.708 | train_tail(strict)=0.0621 train_tail(present)=0.1387 | val acc=0.5321 macroF1=0.1537 conf=0.705 | val_tail(strict)=0.0417 val_tail(present)=0.1095 | best=0.1537@2 | no_improve=0/10 saved
Loaded best model from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_proto_gate_rare_only_k5.pt (Val macroF1=0.1537, epoch=2)
=== Final Test (sklearn-default macro-F1; NOT the reporting convention) ===
Acc=0.5316 | MacroF1=0.1711 | MicroF1=0.5316 | Conf=0.702
TailF1(strict)=0.0361 | TailF1(present)=0.0949 | HeadF1(strict)=0.1703
Final model saved to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/final_model_proto_gate_rare_only_k5.pt

Done: proto_gate_rare_only_k5
{'acc': 0.5316091954022989, 'macro_f1': 0.17110291468924735, 'micro_f1': 0.5316091954022989, 'weighted_f1': 0.4982052974641872, 'avg_conf': 0.7020676136016846, 'best_val_macro_f1': 0.1536896021413393, 'best_epoch': 2, 

train(EVAL-epoch1) acc=0.3983 macroF1=0.0379 conf=0.733 | train_tail(strict)=0.0153 train_tail(present)=0.0342 | val acc=0.3934 macroF1=0.0349 conf=0.733 | val_tail(strict)=0.0167 val_tail(present)=0.0438 | best=0.0349@1 | no_improve=0/10 saved


train(EVAL-epoch2) acc=0.5332 macroF1=0.1509 conf=0.701 | train_tail(strict)=0.0785 train_tail(present)=0.1754 | val acc=0.5181 macroF1=0.1187 conf=0.698 | val_tail(strict)=0.0498 val_tail(present)=0.1307 | best=0.1187@2 | no_improve=0/10 saved
Loaded best model from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/ckpt_proto_concat_rare_only_k5.pt (Val macroF1=0.1187, epoch=2)
=== Final Test (sklearn-default macro-F1; NOT the reporting convention) ===
Acc=0.5149 | MacroF1=0.1317 | MicroF1=0.5149 | Conf=0.695
TailF1(strict)=0.0530 | TailF1(present)=0.1392 | HeadF1(strict)=0.1214
Final model saved to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/checkpoints_test800/final_model_proto_concat_rare_only_k5.pt

Done: proto_concat_rare_only_k5
{'acc': 0.5149425287356322, 'macro_f1': 0.1316979844217011, 'micro_f1': 0.5149425287356322, 'weighted_f1': 0.4696750606880367, 'avg_conf': 0.6949681639671326, 'best_val_macro_f1': 0.11872581429683382, 'best_epoch

## Evaluate (Table 7.8)
Five inference draws per checkpoint, strict macro-F1 over event classes 1..211 and rare-class F1 (training count < 15).

`EVAL_THESIS_CHECKPOINTS = True` re-evaluates the original thesis checkpoints instead (full graph only). Their kNN cache for k=5 is read from Drive; k=3 is rebuilt.

In [14]:
from ddi import evaluate

EVAL_THESIS_CHECKPOINTS = False
if EVAL_THESIS_CHECKPOINTS:
    assert not TEST_MODE, "thesis checkpoints only fit the full graph"
    eval_dir = paths.thesis_warm_ckpt_dir
    import shutil
    if os.path.exists(paths.thesis_knn_cache_k5) and not os.path.exists(os.path.join(KNN_DIR, "knn_cache_k5.pt")):
        shutil.copy(paths.thesis_knn_cache_k5, os.path.join(KNN_DIR, "knn_cache_k5.pt"))
else:
    eval_dir = CKPT_DIR

table_7_8 = evaluate.evaluate_checkpoint_dir(
    g_lg_filt, eval_dir, n_draws=5, device=device, knn_cache_dir=KNN_DIR,
    out_json=os.path.join(paths.warm_dir, f"table_7_8_{RUN_TAG}{'_thesis' if EVAL_THESIS_CHECKPOINTS else ''}.json"))

baseline (no retrieval)                  acc 0.5380  macroF1 0.0651+/-0.0003 | rareF1 0.0000+/-0.0000 | rec 0.0/50
Loaded kNN cache from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/knn_test800/knn_cache_k3.pt: torch.Size([81223, 3])
retrieval: additive, k=3                 acc 0.5121  macroF1 0.0751+/-0.0013 | rareF1 0.0000+/-0.0000 | rec 0.0/50
Loaded kNN cache from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/knn_test800/knn_cache_k5.pt: torch.Size([81223, 5])
retrieval: additive, k=5                 acc 0.5198  macroF1 0.0590+/-0.0006 | rareF1 0.0000+/-0.0000 | rec 0.0/50
Loaded kNN cache from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/knn_test800/knn_cache_k5.pt: torch.Size([81223, 5])
retrieval: gated, k=5                    acc 0.5294  macroF1 0.0988+/-0.0025 | rareF1 0.0535+/-0.0048 | rec 13.4/50
Loaded kNN cache from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/knn_test800/knn_cache_k5.pt: torch.Size([81223, 5])
retrieval: 

In [15]:
print("Thesis Table 7.8 (concat k=5): macro-F1 0.814 +/- 0.004, rare-F1 0.705; "
      "baseline (single draw) 0.729 / 0.369")

Thesis Table 7.8 (concat k=5): macro-F1 0.814 +/- 0.004, rare-F1 0.705; baseline (single draw) 0.729 / 0.369
